# RAG Website Q&A Pipeline


## 1. Setup & Imports

In [1]:
import os
import re
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

print("All imports loaded successfully.")

All imports loaded successfully.


## 2. Load Environment Variables


In [2]:
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
if not groq_key or groq_key == "your_groq_api_key_here":
    raise ValueError("GROQ_API_KEY not set. Add a valid key to the .env file.")

print("Groq API key loaded successfully.")

Groq API key loaded successfully.


## 3. Configuration

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Encoding": "gzip, deflate",
}

TIMEOUT = 15
REMOVE_TAGS = [
    "script", "style", "noscript", "iframe",
    "nav", "footer", "header", "aside", "form", "button"
]

# Chunking config
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Embedding config (free, local)
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# LLM config (free Groq API)
LLM_MODEL = "llama-3.1-8b-instant"
LLM_TEMPERATURE = 0
TOP_K = 4
FAISS_INDEX_DIR = "faiss_index"

# System prompt for RAG
SYSTEM_PROMPT = (
    "You are an information extraction assistant. "
    "Use only the provided webpage content to answer the question. "
    "Do not use external knowledge or make assumptions. "
    "If the information is not explicitly present, respond with: "
    "'Not found in the provided content.' "
    "Keep the response concise and factual."
)
print("Configuration set.")

## 4. Web Scraping

In [4]:
def scrape_website(url: str) -> str:
    """Scrape and clean text content from a website URL."""
    if not url or not url.strip():
        raise ValueError("URL cannot be empty.")

    # Fetch HTML
    response = requests.get(url.strip(), headers=HEADERS, timeout=TIMEOUT)
    response.raise_for_status()

    # Parse HTML
    soup = BeautifulSoup(response.text, "html.parser")

    # Remove non-content tags
    for tag_name in REMOVE_TAGS:
        for tag in soup.find_all(tag_name):
            tag.decompose()

    # Remove elements with navigation/banner roles
    for element in soup.find_all(attrs={"role": ["navigation", "banner", "contentinfo"]}):
        element.decompose()

    # Extract and clean text
    raw_text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", raw_text).strip()

    if not text:
        raise ValueError("No meaningful text could be extracted from the URL.")

    return text

print("scrape_website() defined.")

scrape_website() defined.


### 4.1 Scrape a Website

In [5]:
URL = "https://en.wikipedia.org/wiki/Kanchipuram"

scraped_text = scrape_website(URL)
print(f"Scraped {len(scraped_text):,} characters from: {URL}")
print(f"\nPreview (first 500 chars):\n{scraped_text[:500]}")

Scraped 62,493 characters from: https://en.wikipedia.org/wiki/Kanchipuram

Preview (first 500 chars):
Kanchipuram - Wikipedia Jump to content Coordinates : 12°49′07″N 79°41′41″E ﻿ / ﻿ 12.818500°N 79.694700°E ﻿ / 12.818500; 79.694700 From Wikipedia, the free encyclopedia Municipal Corporation in Tamil Nadu, India For other uses, see Kanchipuram (disambiguation) . This article is about the Municipal Corporation in Tamil Nadu, India. For its namesake district, see Kanchipuram district . "Kanchi" redirects here. For the Bangladeshi actress, see Kanchi (actress) . City in Tamil Nadu, India Kanchipura


## 5. Text Chunking


In [6]:
def chunk_text(text: str) -> list:
    """Split text into overlapping Document chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    documents = splitter.create_documents([text])
    return documents

documents = chunk_text(scraped_text)
print(f"Created {len(documents)} chunks.")
print(f"\nChunk 1 preview ({len(documents[0].page_content)} chars):\n{documents[0].page_content[:300]}")

Created 78 chunks.

Chunk 1 preview (989 chars):
Kanchipuram - Wikipedia Jump to content Coordinates : 12°49′07″N 79°41′41″E ﻿ / ﻿ 12.818500°N 79.694700°E ﻿ / 12.818500; 79.694700 From Wikipedia, the free encyclopedia Municipal Corporation in Tamil Nadu, India For other uses, see Kanchipuram (disambiguation) . This article is about the Municipal C


## 6. Create Embeddings & Build FAISS Vector Store


In [7]:
# Initialize embeddings model (downloads once, then runs locally)
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# Build FAISS vector store from documents
vector_store = FAISS.from_documents(documents, embeddings)

# Save locally for reuse
vector_store.save_local(FAISS_INDEX_DIR)

print(f"FAISS vector store created with {len(documents)} vectors.")
print(f"Index saved to '{FAISS_INDEX_DIR}/' directory.")

FAISS vector store created with 78 vectors.
Index saved to 'faiss_index/' directory.


### 6.1 (Optional) Load Existing FAISS Index

In [8]:
# Uncomment to load from disk instead of rebuilding:
# vector_store = FAISS.load_local(
#     FAISS_INDEX_DIR,
#     embeddings,
#     allow_dangerous_deserialization=True,
# )
# print("FAISS index loaded from disk.")

## 7. Retrieval — Similarity Search


In [9]:
def retrieve_relevant_chunks(vector_store, query: str, k: int = TOP_K) -> list:
    """Run similarity search and return top-k relevant chunks."""
    return vector_store.similarity_search(query, k=k)

# Test retrieval
test_query = "What is retrieval augmented generation?"
retrieved_chunks = retrieve_relevant_chunks(vector_store, test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved_chunks)} chunks:\n")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"--- Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:200])
    print()

Query: What is retrieval augmented generation?
Retrieved 4 chunks:

--- Chunk 1 (887 chars) ---
1911 Encyclopaedia Britannica with Wikisource reference Webarchive template wayback links Articles with short description Short description is different from Wikidata Good articles Use dmy dates from 

--- Chunk 2 (999 chars) ---
ISBN 978-0-14-341517-6 . Economic and political weekly (1995). Economic and political weekly, Volume 30 . Sameeksha Trust. p. 2396. Gopal, Madan (1990). K.S. Gautam (ed.). India through the ages . Pub

--- Chunk 3 (996 chars) ---
Hindu . 19 May 2012 . Retrieved 26 June 2012 . "Deprived of original élan" . The Hindu . 23 June 2011 . Retrieved 26 June 2012 . Dave, Kapil (29 May 2012). "Govt ropes in TCS for much-awaited IIIT pro

--- Chunk 4 (998 chars) ---
Census Commissioner, Ministry of Home Affairs, Government of India. 2013 . Retrieved 26 January 2014 . "Religious places in Kanchipuram" . Tamil Nadu Tourism Development Corporation. 2011 . Retrieved 



## 8. LLM Answer Generation


In [10]:
def generate_answer(query: str, context_docs: list) -> str:
    """Build prompt from context + question, send to Groq LLM, return answer."""
    # Combine chunks into a single context block
    context = "\n\n---\n\n".join(doc.page_content for doc in context_docs)

    # Build the prompt
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human",
         "Context:\n{context}\n\n"
         "Question: {question}\n\n"
         "Answer:"),
    ])

    # Initialize Groq LLM (free API)
    llm = ChatGroq(model=LLM_MODEL, temperature=LLM_TEMPERATURE)

    # Create chain and invoke
    chain = prompt | llm
    response = chain.invoke({"context": context, "question": query})
    return response.content

print("generate_answer() defined.")

generate_answer() defined.


## 9. End-to-End: Ask a Question


In [11]:
def ask_question(query: str, vector_store) -> tuple:
    """Full RAG pipeline: retrieve chunks + generate answer."""
    chunks = retrieve_relevant_chunks(vector_store, query)
    answer = generate_answer(query, chunks)
    return answer, chunks

In [12]:
# --- Ask your question here ---
question = "tell me about temples in kanchipuram?"

answer, chunks = ask_question(question, vector_store)

print(f"Question: {question}\n")
print(f"Answer:\n{answer}\n")
print("=" * 60)
print(f"Retrieved {len(chunks)} context chunks:")
for i, chunk in enumerate(chunks, 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content[:300])

BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama-3.2-3b-preview` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [ ]:
question_2 = "major places to visit in kanchipuram?"

answer_2, chunks_2 = ask_question(question_2, vector_store)
print(f"Question: {question_2}\n")
print(f"Answer:\n{answer_2}")

In [ ]:
question_3 = "over all review about kanchipuram?"

answer_3, chunks_3 = ask_question(question_3, vector_store)
print(f"Question: {question_3}\n")
print(f"Answer:\n{answer_3}")